# MR604 — Full-Phase Combined POC — Corrected Context-Aware

This notebook **Full Phase Combined POC** using the three corrected SLM implementations:

- **Gemma-2B** — R5 LoRA / R6 QLoRA
- **Phi-3 Mini** — R1 LoRA / R2 QLoRA
- **Mistral-7B** — R3 LoRA / R4 QLoRA

It also uses larger-model references:

- **Qwen2.5-14B zero-shot + syllabus context**
- **Qwen2.5-14B 2-shot + syllabus context**


## Full phases

1. Environment and paths
2. Load all six corrected result JSON files
3. Load both corrected Qwen references
4. RQ1 cross-model comparison
5. RQ2 LoRA-vs-QLoRA + seven native question types
6. RQ3 Pareto + 50/25/25 deployment score
7. Verify all six corrected `_CONTEXT` adapters
8. Load official SyllabusQA and syllabus text
9. Reproduce BM25 top-2 context retrieval
10. Context-aware live PEFT model loader
11. Gradio full-phase research + deployment + live QA POC

## Cell 1 — Install compatible packages

In [ ]:


!pip uninstall -y accelerate bitsandbytes >/dev/null 2>&1

!pip install -q --no-cache-dir \
    "transformers==5.16.1" \
    "peft==0.12.0" \
    "datasets" \
    "rank_bm25" \
    "gradio>=5,<7" \
    "sentencepiece" \
    pandas numpy matplotlib

!pip install -q --no-cache-dir --no-deps \
    "accelerate==1.15.0" \
    "bitsandbytes==0.50.2"

print("=" * 72)
print("INSTALLATION COMPLETE")
print("=" * 72)
print("Restart runtime ONCE if Colab requests it.")
print("Then continue from Cell 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 258.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 353.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 380.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 102.7 MB/s eta 0:00:00
INSTALLATION COMPLETE
Restart runtime ONCE if Colab requests it.
Then continue from Cell 2.


## Cell 2 — Imports, GPU, reproducibility

In [ ]:
import os
import gc
import json
import time
import math
import random
import string
import re
import traceback
import importlib.metadata
import subprocess

from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import gradio as gr

import transformers
import accelerate
import peft
import bitsandbytes

from transformers.utils import is_accelerate_available

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

GPU_AVAILABLE = torch.cuda.is_available()

if GPU_AVAILABLE:
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
else:
    GPU_NAME = "CPU only"
    GPU_VRAM_GB = 0.0

print("=" * 72)
print("ENVIRONMENT CHECK")
print("=" * 72)
print("Transformers:", transformers.__version__)
print("Accelerate imported:", accelerate.__version__)
print("Accelerate metadata:", importlib.metadata.version("accelerate"))
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Gradio:", gr.__version__)
print("Transformers detects Accelerate:", is_accelerate_available())
print("GPU:", GPU_NAME)
print(f"GPU VRAM: {GPU_VRAM_GB:.2f} GB")
print("Seed:", SEED)

assert is_accelerate_available(), (
    "Transformers does not detect Accelerate. "
    "Restart runtime once after Cell 1."
)

ENVIRONMENT CHECK
Transformers: 5.16.1
Accelerate imported: 1.15.0
Accelerate metadata: 1.15.0
PEFT: 0.12.0
BitsAndBytes: 0.50.2
Gradio: 6.29.0
Transformers detects Accelerate: True
GPU: NVIDIA A100-SXM4-40GB
GPU VRAM: 39.49 GB
Seed: 42


## Cell 3 — Mount Drive and define thesis paths

In [ ]:
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/MR603_Thesis")
RESULTS_DIR = BASE_DIR / "results"
ADAPTER_DIR = BASE_DIR / "adapters"
CHECKPOINT_DIR = BASE_DIR / "checkpoints"

POC_DIR = RESULTS_DIR / "full_phase_poc_context"
FIGURE_DIR = POC_DIR / "figures"

for p in [RESULTS_DIR, ADAPTER_DIR, CHECKPOINT_DIR, POC_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("THESIS PATHS")
print("=" * 72)
print("BASE_DIR:", BASE_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("POC_DIR:", POC_DIR)

Mounted at /content/drive
THESIS PATHS
BASE_DIR: /content/drive/MyDrive/MR603_Thesis
RESULTS_DIR: /content/drive/MyDrive/MR603_Thesis/results
ADAPTER_DIR: /content/drive/MyDrive/MR603_Thesis/adapters
CHECKPOINT_DIR: /content/drive/MyDrive/MR603_Thesis/checkpoints
POC_DIR: /content/drive/MyDrive/MR603_Thesis/results/full_phase_poc_context


# Phase 1 — Load corrected experimental evidence

This notebook treats the six `_CONTEXT.json` result files as the authoritative source.  
It does **not** hard-code the old blank-context values.

## Cell 4 — Robust JSON metric helpers

In [ ]:
def load_json(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_metric(result, possible_keys, metric_name, required=True):

    for key in possible_keys:

        if key in result and result[key] is not None:
            return float(result[key])

    if required:
        raise KeyError(
            f"{metric_name} not found. "
            f"Available keys: {list(result.keys())}"
        )

    return np.nan


print("✓ JSON helpers ready.")

✓ JSON helpers ready.


## Cell 5 — Load all six corrected SLM results

In [ ]:
CONDITIONS = {
    "gemma_lora": {
        "condition": "R5_Gemma2B_LoRA_CONTEXT",
        "model": "Gemma-2B",
        "method": "LoRA",
        "parameters_b": 2.0,
        "result_file": "R5_gemma2b_lora_CONTEXT.json",
        "adapter": "R5_gemma2b_lora_CONTEXT",
        "base_model": "google/gemma-2b-it",
    },

    "gemma_qlora": {
        "condition": "R6_Gemma2B_QLoRA_CONTEXT",
        "model": "Gemma-2B",
        "method": "QLoRA",
        "parameters_b": 2.0,
        "result_file": "R6_gemma2b_qlora_CONTEXT.json",
        "adapter": "R6_gemma2b_qlora_CONTEXT",
        "base_model": "google/gemma-2b-it",
    },

    "phi_lora": {
        "condition": "R1_Phi3Mini_LoRA_CONTEXT",
        "model": "Phi-3-mini",
        "method": "LoRA",
        "parameters_b": 3.8,
        "result_file": "R1_phi3mini_lora_CONTEXT.json",
        "adapter": "R1_phi3mini_lora_CONTEXT",
        "base_model": "microsoft/Phi-3-mini-4k-instruct",
    },

    "phi_qlora": {
        "condition": "R2_Phi3Mini_QLoRA_CONTEXT",
        "model": "Phi-3-mini",
        "method": "QLoRA",
        "parameters_b": 3.8,
        "result_file": "R2_phi3mini_qlora_CONTEXT.json",
        "adapter": "R2_phi3mini_qlora_CONTEXT",
        "base_model": "microsoft/Phi-3-mini-4k-instruct",
    },

    "mistral_lora": {
        "condition": "R3_Mistral7B_LoRA_CONTEXT",
        "model": "Mistral-7B",
        "method": "LoRA",
        "parameters_b": 7.0,
        "result_file": "R3_mistral7b_lora_CONTEXT.json",
        "adapter": "R3_mistral7b_lora_CONTEXT",
        "base_model": "mistralai/Mistral-7B-Instruct-v0.2",
    },

    "mistral_qlora": {
        "condition": "R4_Mistral7B_QLoRA_CONTEXT",
        "model": "Mistral-7B",
        "method": "QLoRA",
        "parameters_b": 7.0,
        "result_file": "R4_mistral7b_qlora_CONTEXT.json",
        "adapter": "R4_mistral7b_qlora_CONTEXT",
        "base_model": "mistralai/Mistral-7B-Instruct-v0.2",
    },
}

RAW_RESULTS = {}
rows = []
missing = []

for key, spec in CONDITIONS.items():

    path = RESULTS_DIR / spec["result_file"]

    if not path.exists():
        missing.append(spec["result_file"])
        continue

    result = load_json(path)

    RAW_RESULTS[key] = result

    rows.append({
        "Key": key,
        "Condition": spec["condition"],
        "Model": spec["model"],
        "Method": spec["method"],
        "Parameters_B": spec["parameters_b"],

        "EM_percent":
            get_metric(
                result,
                ["em", "EM_percent", "exact_match"],
                f"{key} EM"
            ),

        "F1_percent":
            get_metric(
                result,
                ["f1", "F1_percent"],
                f"{key} F1"
            ),

        "Training_Time_min":
            get_metric(
                result,
                [
                    "training_time_minutes",
                    "Training_Time_min"
                ],
                f"{key} training time"
            ),

        "Avg_Latency_ms":
            get_metric(
                result,
                [
                    "avg_latency_ms",
                    "Avg_Latency_ms",
                    "latency_ms"
                ],
                f"{key} latency"
            ),

        "Peak_Eval_VRAM_GB":
            get_metric(
                result,
                [
                    "peak_vram_gb",
                    "Peak_Eval_VRAM_GB",
                    "peak_eval_vram_gb"
                ],
                f"{key} eval VRAM"
            ),

        "Peak_Train_VRAM_GB":
            get_metric(
                result,
                [
                    "peak_training_vram_gb",
                    "Peak_Train_VRAM_GB"
                ],
                f"{key} training VRAM",
                required=False
            ),

        "Trainable_Params":
            get_metric(
                result,
                [
                    "trainable_parameters",
                    "Trainable_Params"
                ],
                f"{key} trainable parameters",
                required=False
            ),
    })


assert not missing, (
    "Missing corrected SLM result files: "
    + ", ".join(missing)
)

SLM_RESULTS = pd.DataFrame(rows)

assert len(SLM_RESULTS) == 6
assert SLM_RESULTS["Key"].nunique() == 6

for model in ["Gemma-2B", "Phi-3-mini", "Mistral-7B"]:

    model_df = SLM_RESULTS[
        SLM_RESULTS["Model"] == model
    ]

    assert len(model_df) == 2
    assert set(model_df["Method"]) == {"LoRA", "QLoRA"}


print("=" * 92)
print("FINAL CORRECTED SIX-CONDITION SLM MATRIX")
print("=" * 92)

display(
    SLM_RESULTS[
        [
            "Condition",
            "Model",
            "Method",
            "Parameters_B",
            "EM_percent",
            "F1_percent",
            "Training_Time_min",
            "Avg_Latency_ms",
            "Peak_Eval_VRAM_GB",
            "Peak_Train_VRAM_GB"
        ]
    ].round(3)
)

SLM_RESULTS.to_csv(
    POC_DIR / "Final_Six_SLM_CONTEXT_Results.csv",
    index=False
)

print("\n✓ Six corrected SLM conditions loaded.")

FINAL CORRECTED SIX-CONDITION SLM MATRIX


,Condition,Model,Method,Parameters_B,EM_percent,F1_percent,Training_Time_min,Avg_Latency_ms,Peak_Eval_VRAM_GB,Peak_Train_VRAM_GB
0,R5_Gemma2B_LoRA_CONTEXT,Gemma-2B,LoRA,2.0,0.000,17.309,11.342,2410.089,5.121,10.737
1,R6_Gemma2B_QLoRA_CONTEXT,Gemma-2B,QLoRA,2.0,0.000,17.263,16.083,3619.289,3.321,10.092
2,R1_Phi3Mini_LoRA_CONTEXT,Phi-3-mini,LoRA,3.8,0.091,20.030,17.008,3555.800,7.421,8.346
3,R2_Phi3Mini_QLoRA_CONTEXT,Phi-3-mini,QLoRA,3.8,0.000,20.084,21.845,4344.341,3.237,4.448
4,R3_Mistral7B_LoRA_CONTEXT,Mistral-7B,LoRA,7.0,0.000,19.483,23.419,4117.590,13.701,14.897
5,R4_Mistral7B_QLoRA_CONTEXT,Mistral-7B,QLoRA,7.0,0.000,19.357,31.409,6328.417,4.991,6.733



✓ Six corrected SLM conditions loaded.


## Cell 6 — Load corrected Qwen zero-shot and 2-shot references

In [ ]:
QWEN_ZERO_PATH = (
    RESULTS_DIR
    /
    "REF_qwen14b_zeroshot_CONTEXT.json"
)

QWEN_TWO_PATH = (
    RESULTS_DIR
    /
    "REF_qwen14b_2shot_CONTEXT.json"
)

qwen_zero = load_json(
    QWEN_ZERO_PATH
)

qwen_two = load_json(
    QWEN_TWO_PATH
)


def make_qwen_row(label, result):

    return {
        "Reference": label,

        "EM_percent":
            get_metric(
                result,
                [
                    "em",
                    "EM_percent",
                    "exact_match"
                ],
                f"{label} EM"
            ),

        "F1_percent":
            get_metric(
                result,
                [
                    "f1",
                    "F1_percent"
                ],
                f"{label} F1"
            ),

        "Avg_Latency_ms":
            get_metric(
                result,
                [
                    "avg_latency_ms",
                    "Avg_Latency_ms",
                    "latency_ms"
                ],
                f"{label} latency"
            ),

        "Peak_Eval_VRAM_GB":
            get_metric(
                result,
                [
                    "peak_vram_gb",
                    "Peak_Eval_VRAM_GB",
                    "peak_eval_vram_gb",
                    "eval_vram_gb"
                ],
                f"{label} eval VRAM"
            )
    }


QWEN_RESULTS = pd.DataFrame([
    make_qwen_row(
        "Qwen2.5-14B Zero-Shot + Context",
        qwen_zero
    ),

    make_qwen_row(
        "Qwen2.5-14B 2-Shot + Context",
        qwen_two
    )
])

QWEN_ZERO = QWEN_RESULTS.iloc[0]
QWEN_TWO = QWEN_RESULTS.iloc[1]

print("=" * 82)
print("CORRECTED QWEN REFERENCES")
print("=" * 82)

display(
    QWEN_RESULTS.round(3)
)

QWEN_RESULTS.to_csv(
    POC_DIR / "Qwen_CONTEXT_References.csv",
    index=False
)

print("✓ Both corrected Qwen references loaded.")

CORRECTED QWEN REFERENCES


,Reference,EM_percent,F1_percent,Avg_Latency_ms,Peak_Eval_VRAM_GB
0,Qwen2.5-14B Zero-Shot + Context,0.000,17.792,3776.849,27.720
1,Qwen2.5-14B 2-Shot + Context,0.091,18.934,3528.325,29.864


✓ Both corrected Qwen references loaded.


# Phase 2 — RQ1 / RQ2 / RQ3 combined analysis

## Cell 7 — RQ1: SLMs versus both Qwen context references

In [ ]:
RQ1 = SLM_RESULTS.copy()

RQ1[
    "Gap_to_Qwen_ZeroShot_F1_pp"
] = (
    RQ1["F1_percent"]
    -
    QWEN_ZERO["F1_percent"]
)

RQ1[
    "Gap_to_Qwen_2Shot_F1_pp"
] = (
    RQ1["F1_percent"]
    -
    QWEN_TWO["F1_percent"]
)

RQ1[
    "Exceeds_Qwen_ZeroShot"
] = (
    RQ1["F1_percent"]
    >
    QWEN_ZERO["F1_percent"]
)

RQ1[
    "Exceeds_Qwen_2Shot"
] = (
    RQ1["F1_percent"]
    >
    QWEN_TWO["F1_percent"]
)

RQ1 = RQ1.sort_values(
    "F1_percent",
    ascending=False
).reset_index(drop=True)

print("=" * 90)
print("RQ1 — CORRECTED CONTEXT-AWARE CROSS-MODEL PERFORMANCE")
print("=" * 90)

display(
    RQ1[
        [
            "Model",
            "Method",
            "EM_percent",
            "F1_percent",
            "Gap_to_Qwen_ZeroShot_F1_pp",
            "Gap_to_Qwen_2Shot_F1_pp",
            "Exceeds_Qwen_ZeroShot",
            "Exceeds_Qwen_2Shot"
        ]
    ].round(3)
)

print(
    f"\nQwen zero-shot + context F1: "
    f"{QWEN_ZERO['F1_percent']:.3f}%"
)

print(
    f"Qwen 2-shot + context F1: "
    f"{QWEN_TWO['F1_percent']:.3f}%"
)

RQ1.to_csv(
    POC_DIR / "RQ1_CONTEXT_Comparison.csv",
    index=False
)

RQ1 — CORRECTED CONTEXT-AWARE CROSS-MODEL PERFORMANCE


,Model,Method,EM_percent,F1_percent,Gap_to_Qwen_ZeroShot_F1_pp,Gap_to_Qwen_2Shot_F1_pp,Exceeds_Qwen_ZeroShot,Exceeds_Qwen_2Shot
0,Phi-3-mini,QLoRA,0.000,20.084,2.292,1.150,True,True
1,Phi-3-mini,LoRA,0.091,20.030,2.239,1.097,True,True
2,Mistral-7B,LoRA,0.000,19.483,1.691,0.549,True,True
3,Mistral-7B,QLoRA,0.000,19.357,1.566,0.424,True,True
4,Gemma-2B,LoRA,0.000,17.309,-0.483,-1.624,False,False
5,Gemma-2B,QLoRA,0.000,17.263,-0.529,-1.671,False,False



Qwen zero-shot + context F1: 17.792%
Qwen 2-shot + context F1: 18.934%


## Cell 8 — RQ2: paired LoRA vs QLoRA by model

In [ ]:
paired_rows = []

for model in [
    "Gemma-2B",
    "Phi-3-mini",
    "Mistral-7B"
]:

    lora = SLM_RESULTS[
        (SLM_RESULTS["Model"] == model)
        &
        (SLM_RESULTS["Method"] == "LoRA")
    ].iloc[0]

    qlora = SLM_RESULTS[
        (SLM_RESULTS["Model"] == model)
        &
        (SLM_RESULTS["Method"] == "QLoRA")
    ].iloc[0]

    paired_rows.append({
        "Model":
            model,

        "LoRA_F1_percent":
            lora["F1_percent"],

        "QLoRA_F1_percent":
            qlora["F1_percent"],

        "Delta_F1_LoRA_minus_QLoRA_pp":
            (
                lora["F1_percent"]
                -
                qlora["F1_percent"]
            ),

        "LoRA_Latency_ms":
            lora["Avg_Latency_ms"],

        "QLoRA_Latency_ms":
            qlora["Avg_Latency_ms"],

        "LoRA_Eval_VRAM_GB":
            lora["Peak_Eval_VRAM_GB"],

        "QLoRA_Eval_VRAM_GB":
            qlora["Peak_Eval_VRAM_GB"],

        "LoRA_Train_min":
            lora["Training_Time_min"],

        "QLoRA_Train_min":
            qlora["Training_Time_min"],

        "LoRA_Train_VRAM_GB":
            lora["Peak_Train_VRAM_GB"],

        "QLoRA_Train_VRAM_GB":
            qlora["Peak_Train_VRAM_GB"],
    })


RQ2_PAIRED = pd.DataFrame(
    paired_rows
)

print("=" * 90)
print("RQ2 — PAIRED LoRA VS QLoRA")
print("=" * 90)

display(
    RQ2_PAIRED.round(3)
)

RQ2_PAIRED.to_csv(
    POC_DIR / "RQ2_Paired_LoRA_QLoRA_CONTEXT.csv",
    index=False
)

RQ2 — PAIRED LoRA VS QLoRA


,Model,LoRA_F1_percent,QLoRA_F1_percent,Delta_F1_LoRA_minus_QLoRA_pp,LoRA_Latency_ms,QLoRA_Latency_ms,LoRA_Eval_VRAM_GB,QLoRA_Eval_VRAM_GB,LoRA_Train_min,QLoRA_Train_min,LoRA_Train_VRAM_GB,QLoRA_Train_VRAM_GB
0,Gemma-2B,17.309,17.263,0.047,2410.089,3619.289,5.121,3.321,11.342,16.083,10.737,10.092
1,Phi-3-mini,20.030,20.084,-0.053,3555.800,4344.341,7.421,3.237,17.008,21.845,8.346,4.448
2,Mistral-7B,19.483,19.357,0.125,4117.590,6328.417,13.701,4.991,23.419,31.409,14.897,6.733


## Cell 9 — RQ2: seven native question types directly from corrected JSON

In [ ]:
EXPECTED_TYPES = {
    "multi factual",
    "multi reasoning",
    "no answer",
    "single factual",
    "single reasoning",
    "summarization",
    "yes/no"
}

long_rows = []

for key, spec in CONDITIONS.items():

    result = RAW_RESULTS[key]

    strat = result.get(
        "stratified",
        result.get(
            "stratified_em",
            {}
        )
    )

    assert strat, (
        f"No stratified result found for {key}"
    )

    for question_type, values in strat.items():

        long_rows.append({
            "Key":
                key,

            "Model":
                spec["model"],

            "Method":
                spec["method"],

            "Question_Type":
                str(
                    question_type
                ).strip().lower(),

            "N":
                int(
                    values["count"]
                ),

            "EM_percent":
                float(
                    values["exact_match"]
                ),

            "F1_percent":
                float(
                    values["f1"]
                ),
        })


RQ2_LONG = pd.DataFrame(
    long_rows
)

assert len(RQ2_LONG) == 42

for key in CONDITIONS:

    types = set(
        RQ2_LONG[
            RQ2_LONG["Key"] == key
        ]["Question_Type"]
    )

    assert types == EXPECTED_TYPES, (
        f"{key} question types do not match expected 7 types."
    )


lora_qt = (
    RQ2_LONG[
        RQ2_LONG["Method"] == "LoRA"
    ][
        [
            "Model",
            "Question_Type",
            "N",
            "EM_percent",
            "F1_percent"
        ]
    ]
    .rename(
        columns={
            "EM_percent":
                "LoRA_EM_percent",
            "F1_percent":
                "LoRA_F1_percent"
        }
    )
)

qlora_qt = (
    RQ2_LONG[
        RQ2_LONG["Method"] == "QLoRA"
    ][
        [
            "Model",
            "Question_Type",
            "EM_percent",
            "F1_percent"
        ]
    ]
    .rename(
        columns={
            "EM_percent":
                "QLoRA_EM_percent",
            "F1_percent":
                "QLoRA_F1_percent"
        }
    )
)

RQ2_TYPES = lora_qt.merge(
    qlora_qt,
    on=[
        "Model",
        "Question_Type"
    ],
    how="inner"
)

RQ2_TYPES[
    "Delta_F1_LoRA_minus_QLoRA_pp"
] = (
    RQ2_TYPES[
        "LoRA_F1_percent"
    ]
    -
    RQ2_TYPES[
        "QLoRA_F1_percent"
    ]
)

assert len(RQ2_TYPES) == 21

print("=" * 94)
print("RQ2 — NATIVE SYLLABUSQA QUESTION TYPES")
print("=" * 94)

display(
    RQ2_TYPES.sort_values(
        [
            "Model",
            "Question_Type"
        ]
    ).round(3)
)

RQ2_TYPES.to_csv(
    POC_DIR / "RQ2_QuestionType_CONTEXT.csv",
    index=False
)

RQ2 — NATIVE SYLLABUSQA QUESTION TYPES


,Model,Question_Type,N,LoRA_EM_percent,LoRA_F1_percent,QLoRA_EM_percent,QLoRA_F1_percent,Delta_F1_LoRA_minus_QLoRA_pp
0,Gemma-2B,multi factual,158,0.000,27.368,0.0,26.985,0.383
1,Gemma-2B,multi reasoning,158,0.000,24.892,0.0,24.800,0.091
2,Gemma-2B,no answer,155,0.000,2.811,0.0,3.098,-0.288
3,Gemma-2B,single factual,158,0.000,15.743,0.0,16.395,-0.652
4,Gemma-2B,single reasoning,158,0.000,21.509,0.0,20.876,0.632
5,Gemma-2B,summarization,158,0.000,25.771,0.0,25.288,0.483
6,Gemma-2B,yes/no,158,0.000,2.796,0.0,3.126,-0.330
14,Mistral-7B,multi factual,158,0.000,30.545,0.0,29.867,0.678
15,Mistral-7B,multi reasoning,158,0.000,27.359,0.0,25.982,1.377
16,Mistral-7B,no answer,155,0.000,5.646,0.0,6.619,-0.973


## Cell 10 — RQ3: Pareto analysis

In [ ]:
def is_dominated(i, df):

    a = df.loc[i]

    for j in df.index:

        if j == i:
            continue

        b = df.loc[j]

        no_worse = (
            b["F1_percent"]
            >=
            a["F1_percent"]
            and
            b["Avg_Latency_ms"]
            <=
            a["Avg_Latency_ms"]
            and
            b["Peak_Eval_VRAM_GB"]
            <=
            a["Peak_Eval_VRAM_GB"]
        )

        strictly_better = (
            b["F1_percent"]
            >
            a["F1_percent"]
            or
            b["Avg_Latency_ms"]
            <
            a["Avg_Latency_ms"]
            or
            b["Peak_Eval_VRAM_GB"]
            <
            a["Peak_Eval_VRAM_GB"]
        )

        if no_worse and strictly_better:
            return True

    return False


DEPLOY = SLM_RESULTS.copy()

DEPLOY["Pareto_Efficient"] = [
    not is_dominated(
        i,
        DEPLOY
    )
    for i in DEPLOY.index
]

print("=" * 86)
print("RQ3 — PARETO ANALYSIS")
print("=" * 86)

display(
    DEPLOY[
        [
            "Condition",
            "F1_percent",
            "Avg_Latency_ms",
            "Peak_Eval_VRAM_GB",
            "Pareto_Efficient"
        ]
    ].round(3)
)

RQ3 — PARETO ANALYSIS


,Condition,F1_percent,Avg_Latency_ms,Peak_Eval_VRAM_GB,Pareto_Efficient
0,R5_Gemma2B_LoRA_CONTEXT,17.309,2410.089,5.121,True
1,R6_Gemma2B_QLoRA_CONTEXT,17.263,3619.289,3.321,True
2,R1_Phi3Mini_LoRA_CONTEXT,20.030,3555.800,7.421,True
3,R2_Phi3Mini_QLoRA_CONTEXT,20.084,4344.341,3.237,True
4,R3_Mistral7B_LoRA_CONTEXT,19.483,4117.590,13.701,False
5,R4_Mistral7B_QLoRA_CONTEXT,19.357,6328.417,4.991,False


## Cell 11 — RQ3: secondary 50/25/25 deployment score

In [ ]:
def benefit_norm(series):

    series = pd.to_numeric(
        series
    )

    if series.max() == series.min():
        return pd.Series(
            np.ones(
                len(series)
            ),
            index=series.index
        )

    return (
        series - series.min()
    ) / (
        series.max() - series.min()
    )


def cost_norm(series):

    series = pd.to_numeric(
        series
    )

    if series.max() == series.min():
        return pd.Series(
            np.ones(
                len(series)
            ),
            index=series.index
        )

    return (
        series.max() - series
    ) / (
        series.max() - series.min()
    )


DEPLOY[
    "F1_Norm"
] = benefit_norm(
    DEPLOY["F1_percent"]
)

DEPLOY[
    "Latency_Eff"
] = cost_norm(
    DEPLOY["Avg_Latency_ms"]
)

DEPLOY[
    "Eval_VRAM_Eff"
] = cost_norm(
    DEPLOY["Peak_Eval_VRAM_GB"]
)

DEPLOY[
    "Deployment_Score"
] = 100 * (
    0.50 * DEPLOY["F1_Norm"]
    +
    0.25 * DEPLOY["Latency_Eff"]
    +
    0.25 * DEPLOY["Eval_VRAM_Eff"]
)

DEPLOY = DEPLOY.sort_values(
    "Deployment_Score",
    ascending=False
).reset_index(drop=True)

DEPLOY.insert(
    0,
    "Rank",
    range(
        1,
        len(DEPLOY) + 1
    )
)

print("=" * 90)
print("RQ3 — SECONDARY DEPLOYMENT SCORE")
print("=" * 90)

display(
    DEPLOY[
        [
            "Rank",
            "Condition",
            "F1_percent",
            "Avg_Latency_ms",
            "Peak_Eval_VRAM_GB",
            "Peak_Train_VRAM_GB",
            "Pareto_Efficient",
            "Deployment_Score"
        ]
    ].round(3)
)

DEPLOY.to_csv(
    POC_DIR / "RQ3_Deployment_CONTEXT.csv",
    index=False
)

print(
    "\n50/25/25 = 50% F1 + 25% latency efficiency + "
    "25% evaluation-VRAM efficiency."
)

print(
    "Training VRAM is displayed descriptively but is not added "
    "to the original composite formula."
)

RQ3 — SECONDARY DEPLOYMENT SCORE


,Rank,Condition,F1_percent,Avg_Latency_ms,Peak_Eval_VRAM_GB,Peak_Train_VRAM_GB,Pareto_Efficient,Deployment_Score
0,1,R2_Phi3Mini_QLoRA_CONTEXT,20.084,4344.341,3.237,4.448,True,87.659
1,2,R1_Phi3Mini_LoRA_CONTEXT,20.030,3555.800,7.421,8.346,True,81.746
2,3,R4_Mistral7B_QLoRA_CONTEXT,19.357,6328.417,4.991,6.733,False,57.933
3,4,R3_Mistral7B_LoRA_CONTEXT,19.483,4117.590,13.701,14.897,False,53.450
4,5,R5_Gemma2B_LoRA_CONTEXT,17.309,2410.089,5.121,10.737,True,46.325
5,6,R6_Gemma2B_QLoRA_CONTEXT,17.263,3619.289,3.321,10.092,True,42.086



50/25/25 = 50% F1 + 25% latency efficiency + 25% evaluation-VRAM efficiency.
Training VRAM is displayed descriptively but is not added to the original composite formula.


## Cell 12 — Deployment priority helper

In [ ]:
PRIORITIES = [
    "Balanced deployment",
    "Maximum QA performance",
    "Lowest inference latency",
    "Lowest evaluation VRAM",
    "Shortest training time",
    "Lowest training VRAM",
]


def recommend(priority):

    if priority == "Maximum QA performance":

        return DEPLOY.sort_values(
            "F1_percent",
            ascending=False
        ).iloc[0]

    if priority == "Lowest inference latency":

        return DEPLOY.sort_values(
            "Avg_Latency_ms",
            ascending=True
        ).iloc[0]

    if priority == "Lowest evaluation VRAM":

        return DEPLOY.sort_values(
            "Peak_Eval_VRAM_GB",
            ascending=True
        ).iloc[0]

    if priority == "Shortest training time":

        return DEPLOY.sort_values(
            "Training_Time_min",
            ascending=True
        ).iloc[0]

    if priority == "Lowest training VRAM":

        valid = DEPLOY.dropna(
            subset=[
                "Peak_Train_VRAM_GB"
            ]
        )

        return valid.sort_values(
            "Peak_Train_VRAM_GB",
            ascending=True
        ).iloc[0]

    return DEPLOY.sort_values(
        "Deployment_Score",
        ascending=False
    ).iloc[0]


for p in PRIORITIES:

    row = recommend(p)

    print(
        f"{p:30s} -> "
        f"{row['Condition']:32s} | "
        f"F1={row['F1_percent']:.3f}% | "
        f"Latency={row['Avg_Latency_ms']:.1f} ms | "
        f"Eval VRAM={row['Peak_Eval_VRAM_GB']:.3f} GB"
    )

Balanced deployment            -> R2_Phi3Mini_QLoRA_CONTEXT        | F1=20.084% | Latency=4344.3 ms | Eval VRAM=3.237 GB
Maximum QA performance         -> R2_Phi3Mini_QLoRA_CONTEXT        | F1=20.084% | Latency=4344.3 ms | Eval VRAM=3.237 GB
Lowest inference latency       -> R5_Gemma2B_LoRA_CONTEXT          | F1=17.309% | Latency=2410.1 ms | Eval VRAM=5.121 GB
Lowest evaluation VRAM         -> R2_Phi3Mini_QLoRA_CONTEXT        | F1=20.084% | Latency=4344.3 ms | Eval VRAM=3.237 GB
Shortest training time         -> R5_Gemma2B_LoRA_CONTEXT          | F1=17.309% | Latency=2410.1 ms | Eval VRAM=5.121 GB
Lowest training VRAM           -> R2_Phi3Mini_QLoRA_CONTEXT        | F1=20.084% | Latency=4344.3 ms | Eval VRAM=3.237 GB


# Phase 3 — Verify corrected `_CONTEXT` adapters

## Cell 13 — Resolve all six corrected adapters

In [ ]:
def has_adapter_files(path):

    path = Path(path)

    if not path.exists():
        return False

    config_ok = (
        path
        /
        "adapter_config.json"
    ).exists()

    weights_ok = (
        (
            path
            /
            "adapter_model.safetensors"
        ).exists()
        or
        (
            path
            /
            "adapter_model.bin"
        ).exists()
    )

    return config_ok and weights_ok


def latest_checkpoint(root):

    root = Path(root)

    if not root.exists():
        return None

    found = []

    for p in root.glob(
        "checkpoint-*"
    ):

        if has_adapter_files(p):

            try:
                step = int(
                    p.name.split(
                        "-"
                    )[-1]
                )
            except Exception:
                step = -1

            found.append(
                (
                    step,
                    p
                )
            )

    if not found:
        return None

    return max(
        found,
        key=lambda x:
            x[0]
    )[1]


RESOLVED_ADAPTERS = {}

print("=" * 88)
print("CORRECTED CONTEXT ADAPTER CHECK")
print("=" * 88)

for key, spec in CONDITIONS.items():

    direct = (
        ADAPTER_DIR
        /
        spec["adapter"]
    )

    if has_adapter_files(
        direct
    ):

        resolved = direct
        source = "final adapter"

    else:

        checkpoint = latest_checkpoint(
            CHECKPOINT_DIR
            /
            spec["adapter"]
        )

        if checkpoint is not None:
            resolved = checkpoint
            source = "latest checkpoint"
        else:
            resolved = None
            source = "missing"

    RESOLVED_ADAPTERS[
        key
    ] = resolved

    print(
        f"{'FOUND' if resolved else 'MISSING':7s} | "
        f"{spec['condition']:34s} | "
        f"{source}"
    )

    if resolved:
        print(
            "         ",
            resolved
        )

print(
    "\nResearch analysis remains valid even if a live adapter is missing; "
    "that condition will simply be unavailable in the live POC."
)

CORRECTED CONTEXT ADAPTER CHECK
FOUND   | R5_Gemma2B_LoRA_CONTEXT            | final adapter
          /content/drive/MyDrive/MR603_Thesis/adapters/R5_gemma2b_lora_CONTEXT
FOUND   | R6_Gemma2B_QLoRA_CONTEXT           | final adapter
          /content/drive/MyDrive/MR603_Thesis/adapters/R6_gemma2b_qlora_CONTEXT
FOUND   | R1_Phi3Mini_LoRA_CONTEXT           | final adapter
          /content/drive/MyDrive/MR603_Thesis/adapters/R1_phi3mini_lora_CONTEXT
FOUND   | R2_Phi3Mini_QLoRA_CONTEXT          | final adapter
          /content/drive/MyDrive/MR603_Thesis/adapters/R2_phi3mini_qlora_CONTEXT
FOUND   | R3_Mistral7B_LoRA_CONTEXT          | final adapter
          /content/drive/MyDrive/MR603_Thesis/adapters/R3_mistral7b_lora_CONTEXT
FOUND   | R4_Mistral7B_QLoRA_CONTEXT         | final adapter
          /content/drive/MyDrive/MR603_Thesis/adapters/R4_mistral7b_qlora_CONTEXT

Research analysis remains valid even if a live adapter is missing; that condition will simply be unavailable in the li

# Phase 4 — Load official SyllabusQA and reproduce corrected BM25 context retrieval

## Cell 14 — Clone/load SyllabusQA

In [ ]:
from rank_bm25 import BM25Okapi

REPO_DIR = Path(
    "/content/syllabusqa"
)

if not REPO_DIR.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "-q",
            "https://github.com/umass-ml4ed/syllabusqa.git",
            str(
                REPO_DIR
            )
        ],
        check=True
    )

TEST_JSON = (
    REPO_DIR
    /
    "data"
    /
    "dataset_split"
    /
    "test.json"
)

TEXT_DIR = (
    REPO_DIR
    /
    "syllabi"
    /
    "syllabi_redacted"
    /
    "text"
)

assert TEST_JSON.exists()
assert TEXT_DIR.exists()

with open(
    TEST_JSON,
    "r",
    encoding="utf-8"
) as f:

    raw_test = json.load(
        f
    )


def locate_examples(obj):

    if isinstance(
        obj,
        list
    ):

        if (
            obj
            and
            isinstance(
                obj[0],
                dict
            )
            and
            "question"
            in obj[0]
        ):

            return obj

        for item in obj:

            result = locate_examples(
                item
            )

            if result:
                return result

    if isinstance(
        obj,
        dict
    ):

        for value in obj.values():

            result = locate_examples(
                value
            )

            if result:
                return result

    return None


TEST_EXAMPLES = locate_examples(
    raw_test
)

assert TEST_EXAMPLES is not None
assert len(TEST_EXAMPLES) == 1103

syllabus_map = {}

for path in TEXT_DIR.glob(
    "*.txt"
):

    if (
        path.name.lower()
        ==
        "requirements.txt"
    ):
        continue

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as f:

        syllabus_map[
            path.stem
        ] = f.read().strip()


print("Test examples:", len(TEST_EXAMPLES))
print("Syllabus texts:", len(syllabus_map))

assert len(syllabus_map) >= 63

print("✓ SyllabusQA and source syllabus texts loaded.")

Test examples: 1103
Syllabus texts: 63
✓ SyllabusQA and source syllabus texts loaded.


## Cell 15 — Same corrected BM25 retrieval used in the experiments

In [ ]:
CHUNK_WORDS = 90
CHUNK_OVERLAP = 30
TOP_K = 2


def split_syllabus(
    text,
    chunk_words=
        CHUNK_WORDS,
    overlap=
        CHUNK_OVERLAP
):

    words = str(
        text
    ).split()

    step = (
        chunk_words
        -
        overlap
    )

    chunks = []

    for start in range(
        0,
        len(words),
        step
    ):

        chunk = " ".join(
            words[
                start:
                start
                +
                chunk_words
            ]
        ).strip()

        if chunk:
            chunks.append(
                chunk
            )

    return chunks


def retrieve_context(
    syllabus_name,
    question,
    top_k=
        TOP_K
):

    text = syllabus_map.get(
        str(
            syllabus_name
        ),
        ""
    )

    if not text:

        raise KeyError(
            f"No syllabus text found for {syllabus_name}"
        )

    chunks = split_syllabus(
        text
    )

    tokenized_chunks = [
        re.findall(
            r"\w+",
            c.lower()
        )
        for c in chunks
    ]

    bm25 = BM25Okapi(
        tokenized_chunks
    )

    query = re.findall(
        r"\w+",
        str(
            question
        ).lower()
    )

    scores = bm25.get_scores(
        query
    )

    best = sorted(
        range(
            len(scores)
        ),
        key=lambda i:
            scores[i],
        reverse=True
    )[
        :top_k
    ]

    return "\n".join(
        chunks[i]
        for i in best
    )


def get_question(ex):
    return str(
        ex.get(
            "question",
            ""
        )
    ).strip()


def get_question_type(ex):
    return str(
        ex.get(
            "question_type",
            "unknown"
        )
    ).strip()


def get_syllabus_name(ex):
    return str(
        ex.get(
            "syllabus_name",
            ""
        )
    ).strip()


def get_reference(ex):

    answer = ex.get(
        "answer",
        ""
    )

    if isinstance(
        answer,
        dict
    ):

        texts = answer.get(
            "text",
            []
        )

        return (
            str(
                texts[0]
            )
            if texts
            else ""
        )

    if isinstance(
        answer,
        list
    ):

        return (
            str(
                answer[0]
            )
            if answer
            else ""
        )

    return str(
        answer
    )


sample = TEST_EXAMPLES[0]

sample_context = retrieve_context(
    get_syllabus_name(
        sample
    ),
    get_question(
        sample
    )
)

assert sample_context.strip()

print("=" * 72)
print("BM25 CONTEXT RETRIEVAL READY")
print("=" * 72)
print("Chunk words:", CHUNK_WORDS)
print("Overlap:", CHUNK_OVERLAP)
print("Top-K:", TOP_K)
print("\nSample context:\n")
print(sample_context[:1000])

BM25 CONTEXT RETRIEVAL READY
Chunk words: 90
Overlap: 30
Top-K: 2

Sample context:

Extra Credit: Four extra credit opportunities will be available during the semester. Three of these are exam reflections and the fourth is the end-of-course survey. Each one has a maximum value of 0.3 percentage points towards your final grade. Additional details will be provided as each assignment becomes available. Final Grading Scale: Final grades are rounded to the tenth and then assigned according to this scale. A 93.00 100.00% B-80.00 82.99% D+ 67.00 69.99% A-90.00 92.99% C+ 77.00 79.99% D 60.00 66.99% B+ 87.00 89.99% C 73.00 76.99% F <59.99
miss some assignments and classes are built into the course and should cover most circumstances, but if you have an unexpected situation or assignment extensions are part of your accommodations, contact me as soon as possible to discuss an extension. Note: Because of the course flexibility, extension requests should be infrequent. Student Learning Accommodatio

## Cell 16 — Select 14 fixed POC questions: two from each native type

In [ ]:
groups = defaultdict(
    list
)

for ex in TEST_EXAMPLES:

    qtype = (
        get_question_type(
            ex
        )
        .strip()
        .lower()
    )

    groups[
        qtype
    ].append(
        ex
    )


TARGET_TYPES = [
    "single factual",
    "multi factual",
    "single reasoning",
    "multi reasoning",
    "yes/no",
    "summarization",
    "no answer"
]

rng = random.Random(
    SEED
)

POC_EXAMPLES = []

for qtype in TARGET_TYPES:

    available = (
        groups[
            qtype
        ].copy()
    )

    assert len(
        available
    ) >= 2

    rng.shuffle(
        available
    )

    POC_EXAMPLES.extend(
        available[:2]
    )


QUESTION_MAP = {}

QUESTION_LABELS = []

for i, ex in enumerate(
    POC_EXAMPLES,
    start=1
):

    q = get_question(
        ex
    )

    qt = get_question_type(
        ex
    )

    short_q = (
        q
        if len(q) <= 90
        else q[:87] + "..."
    )

    label = (
        f"{i:02d} | "
        f"{qt} | "
        f"{short_q}"
    )

    QUESTION_LABELS.append(
        label
    )

    QUESTION_MAP[
        label
    ] = ex


assert len(
    QUESTION_LABELS
) == 14

for label in QUESTION_LABELS:
    print(label)

print(
    "\n✓ 14 deterministic POC questions ready."
)

01 | single factual | When is the extra credit assignment due?
02 | single factual | How many exams do we have to take?
03 | multi factual | Where is the professor's office located, and what are the office hours?
04 | multi factual | What is TMSIDK Data analysis project?
05 | single reasoning | What happens if I forget to post a prompt response on the forum?
06 | single reasoning | If I miss an individual assignment will I still be able to make an A?
07 | multi reasoning | If I'm a graduate student and I get a 72% in the course, will I still pass?
08 | multi reasoning | I won't be able to access Moodle before March 1st. Will I still be able to submit Probl...
09 | yes/no | Did the time and date provided in the syllabus will change based on the difficulty of t...
10 | yes/no | Do we have online support?
11 | summarization | Can I get more information on how the CILOs are assessed?
12 | summarization | I've heard about the exercise sessions in this course, can you explain what those are?

# Phase 5 — Context-aware live PEFT inference

## Cell 17 — Hugging Face login if needed

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:

    token = userdata.get(
        "HF_TOKEN"
    )

    if token:
        login(
            token=token
        )
        print("✓ Hugging Face login successful from Colab secret.")
    else:
        print(
            "HF_TOKEN secret not found. "
            "Gemma may require authentication."
        )

except Exception as e:

    print(
        "Hugging Face login skipped:",
        e
    )

Hugging Face login skipped: Secret HF_TOKEN does not exist.


## Cell 18 — Robust native model loader for corrected adapters

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Phi3ForCausalLM,
    MistralForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel

MAX_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 64


class ContextAwarePOC:

    def __init__(
        self
    ):

        self.model = None
        self.tokenizer = None
        self.loaded_key = None
        self.load_seconds = 0.0


    def unload(
        self
    ):

        self.model = None
        self.tokenizer = None
        self.loaded_key = None
        self.load_seconds = 0.0

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    @staticmethod
    def compute_dtype():

        if (
            torch.cuda.is_available()
            and
            torch.cuda.is_bf16_supported()
        ):

            return torch.bfloat16

        return torch.float16


    @staticmethod
    def make_prompt(
        context,
        question
    ):

        context = str(
            context
        ).strip()

        question = str(
            question
        ).strip()

        if not context:
            raise ValueError(
                "Corrected POC requires non-empty syllabus context."
            )

        return (
            f"Syllabus context:\n"
            f"{context}\n\n"
            f"Question:\n"
            f"{question}\n\n"
            f"Answer:"
        )


    def load(
        self,
        key
    ):

        if not torch.cuda.is_available():

            raise RuntimeError(
                "CUDA GPU required for live PEFT inference."
            )

        if key not in CONDITIONS:

            raise KeyError(
                key
            )

        if (
            self.loaded_key == key
            and
            self.model is not None
        ):

            return

        adapter_path = RESOLVED_ADAPTERS.get(
            key
        )

        if adapter_path is None:

            raise FileNotFoundError(
                f"No corrected adapter available for {key}."
            )

        self.unload()

        spec = CONDITIONS[
            key
        ]

        dtype = self.compute_dtype()

        tokenizer = AutoTokenizer.from_pretrained(
            spec[
                "base_model"
            ]
        )

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        tokenizer.padding_side = "right"

        start = time.perf_counter()


        if spec["method"] == "QLoRA":

            quant_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=dtype
            )

        else:

            quant_cfg = None


        common = {
            "device_map":
                {"": 0},
            "low_cpu_mem_usage":
                True,
        }

        if quant_cfg is not None:

            common[
                "quantization_config"
            ] = quant_cfg

        else:

            common[
                "dtype"
            ] = dtype


        # Phi-3: force native class to avoid old remote-code rope_scaling error.
        if key.startswith(
            "phi_"
        ):

            base = Phi3ForCausalLM.from_pretrained(
                spec[
                    "base_model"
                ],
                attn_implementation=
                    "sdpa",
                **common
            )


        # Mistral: use native class, same family as corrected notebook.
        elif key.startswith(
            "mistral_"
        ):

            # On smaller GPUs, full LoRA may require automatic CPU offload.
            if (
                spec["method"] == "LoRA"
                and
                GPU_VRAM_GB < 20
            ):

                common.pop(
                    "device_map",
                    None
                )

                common[
                    "device_map"
                ] = "auto"

                common[
                    "max_memory"
                ] = {
                    0:
                        "14GiB",
                    "cpu":
                        "40GiB"
                }

            base = MistralForCausalLM.from_pretrained(
                spec[
                    "base_model"
                ],
                attn_implementation=
                    "sdpa",
                **common
            )


        # Gemma
        else:

            base = AutoModelForCausalLM.from_pretrained(
                spec[
                    "base_model"
                ],
                **common
            )


        model = PeftModel.from_pretrained(
            base,
            str(
                adapter_path
            )
        )

        model.eval()

        self.model = model
        self.tokenizer = tokenizer
        self.loaded_key = key
        self.load_seconds = (
            time.perf_counter()
            -
            start
        )


    def answer(
        self,
        key,
        question,
        context
    ):

        self.load(
            key
        )

        prompt = self.make_prompt(
            context,
            question
        )

        inputs = self.tokenizer(
            prompt,
            return_tensors=
                "pt",
            max_length=
                MAX_SEQ_LENGTH,
            truncation=
                True,
            padding=
                False
        )

        try:

            device = (
                self.model
                .get_input_embeddings()
                .weight
                .device
            )

        except Exception:

            device = next(
                self.model.parameters()
            ).device

        inputs = {
            k:
                v.to(
                    device
                )
            for k, v
            in inputs.items()
        }

        input_len = (
            inputs[
                "input_ids"
            ].shape[1]
        )

        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

        start = time.perf_counter()

        with torch.inference_mode():

            output = self.model.generate(
                **inputs,
                max_new_tokens=
                    MAX_NEW_TOKENS,
                do_sample=
                    False,
                num_beams=
                    1,
                pad_token_id=
                    self.tokenizer.pad_token_id,
                eos_token_id=
                    self.tokenizer.eos_token_id,
                use_cache=
                    True
            )

        torch.cuda.synchronize()

        latency_ms = (
            time.perf_counter()
            -
            start
        ) * 1000

        new_tokens = output[
            0
        ][
            input_len:
        ]

        answer = self.tokenizer.decode(
            new_tokens,
            skip_special_tokens=
                True
        ).strip()

        peak_vram = (
            torch.cuda.max_memory_allocated()
            /
            1024**3
        )

        return {
            "answer":
                answer,
            "latency_ms":
                float(
                    latency_ms
                ),
            "peak_vram_gb":
                float(
                    peak_vram
                ),
            "load_seconds":
                float(
                    self.load_seconds
                ),
            "prompt":
                prompt,
        }


LIVE_POC = ContextAwarePOC()

print("✓ Context-aware live POC loader ready.")

✓ Context-aware live POC loader ready.


## Cell 19 — Optional live smoke test

In [ ]:
AVAILABLE_KEYS = [
    key
    for key, value
    in RESOLVED_ADAPTERS.items()
    if value is not None
]

if not torch.cuda.is_available():

    print(
        "SKIPPED — no GPU attached."
    )

elif not AVAILABLE_KEYS:

    print(
        "SKIPPED — no corrected adapters available."
    )

else:

    smoke_key = (
        "phi_qlora"
        if "phi_qlora"
        in AVAILABLE_KEYS
        else AVAILABLE_KEYS[0]
    )

    ex = QUESTION_MAP[
        QUESTION_LABELS[0]
    ]

    q = get_question(
        ex
    )

    c = retrieve_context(
        get_syllabus_name(
            ex
        ),
        q
    )

    try:

        result = LIVE_POC.answer(
            smoke_key,
            q,
            c
        )

        print("=" * 72)
        print("LIVE CONTEXT-AWARE SMOKE TEST PASSED")
        print("=" * 72)
        print("Condition:", CONDITIONS[smoke_key]["condition"])
        print("Question:", q)
        print("Answer:", result["answer"])
        print("Latency (ms):", round(result["latency_ms"], 2))
        print("Peak live VRAM (GB):", round(result["peak_vram_gb"], 3))
        print("Load seconds:", round(result["load_seconds"], 2))

    except Exception:

        print(
            "LIVE SMOKE TEST FAILED"
        )

        traceback.print_exc()

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

LIVE CONTEXT-AWARE SMOKE TEST PASSED
Condition: R2_Phi3Mini_QLoRA_CONTEXT
Question: When is the extra credit assignment due?
Answer: The extra credit assignment is due on April 28th at 9PM. 

Note: The extra credit assignment is worth up to 1 point on the students final grade in this class. 

The extra credit assignment is worth up to 1 point on the students final grade in this class
Latency (ms): 5284.79
Peak live VRAM (GB): 2.378
Load seconds: 32.4


# Phase 6 — Full combined Gradio POC

## Cell 20 — Dashboard helper functions

In [ ]:
MODEL_LABELS = {
    key:
        (
            f"{spec['model']} — "
            f"{spec['method']} + Context"
        )
    for key, spec
    in CONDITIONS.items()
}

LABEL_TO_KEY = {
    label:
        key
    for key, label
    in MODEL_LABELS.items()
}

DEFAULT_LABEL = MODEL_LABELS[
    "phi_qlora"
]


def selected_condition_summary(
    label
):

    key = LABEL_TO_KEY[
        label
    ]

    row = SLM_RESULTS[
        SLM_RESULTS[
            "Key"
        ]
        ==
        key
    ].iloc[0]

    zero_gap = (
        row[
            "F1_percent"
        ]
        -
        QWEN_ZERO[
            "F1_percent"
        ]
    )

    two_gap = (
        row[
            "F1_percent"
        ]
        -
        QWEN_TWO[
            "F1_percent"
        ]
    )

    return f"""
### Selected corrected condition

**{row['Condition']}**

- F1: **{row['F1_percent']:.3f}%**
- EM: **{row['EM_percent']:.3f}%**
- Training time: **{row['Training_Time_min']:.2f} min**
- Average inference latency: **{row['Avg_Latency_ms']:.1f} ms/sample**
- Peak evaluation VRAM: **{row['Peak_Eval_VRAM_GB']:.3f} GB**
- Peak training VRAM: **{row['Peak_Train_VRAM_GB']:.3f} GB**

### Comparison with corrected Qwen references

- Gap to Qwen zero-shot + context: **{zero_gap:+.3f} pp**
- Gap to Qwen 2-shot + context: **{two_gap:+.3f} pp**
"""


def deployment_summary(
    priority
):

    row = recommend(
        priority
    )

    return f"""
### Deployment criterion: {priority}

**Selected by this criterion:**
**{row['Condition']}**

- F1: **{row['F1_percent']:.3f}%**
- Latency: **{row['Avg_Latency_ms']:.1f} ms/sample**
- Peak evaluation VRAM: **{row['Peak_Eval_VRAM_GB']:.3f} GB**
- Training time: **{row['Training_Time_min']:.2f} min**
- Peak training VRAM: **{row['Peak_Train_VRAM_GB']:.3f} GB**
- Pareto efficient: **{row['Pareto_Efficient']}**
- Secondary deployment score: **{row['Deployment_Score']:.2f}**
"""


def load_question_details(
    label
):

    ex = QUESTION_MAP[
        label
    ]

    q = get_question(
        ex
    )

    qt = get_question_type(
        ex
    )

    sn = get_syllabus_name(
        ex
    )

    ref = get_reference(
        ex
    )

    context = retrieve_context(
        sn,
        q
    )

    info = (
        f"Syllabus: {sn}\n"
        f"Question type: {qt}\n"
        f"Reference answer: {ref}\n"
        f"Context retrieval: BM25 top-{TOP_K}, "
        f"{CHUNK_WORDS}-word chunks, "
        f"{CHUNK_OVERLAP}-word overlap"
    )

    return (
        q,
        context,
        info
    )


def run_live(
    model_label,
    question,
    context
):

    key = LABEL_TO_KEY[
        model_label
    ]

    if not torch.cuda.is_available():

        return (
            "",
            0.0,
            0.0,
            0.0,
            "",
            "No CUDA GPU attached. Frozen research evidence remains available."
        )

    if (
        RESOLVED_ADAPTERS.get(
            key
        )
        is None
    ):

        return (
            "",
            0.0,
            0.0,
            0.0,
            "",
            "Corrected context adapter is not available for this condition."
        )

    if not str(
        context
    ).strip():

        return (
            "",
            0.0,
            0.0,
            0.0,
            "",
            "Corrected POC requires non-empty syllabus context."
        )

    try:

        result = LIVE_POC.answer(
            key,
            question,
            context
        )

        status = (
            "Live corrected-context inference completed. "
            "These live diagnostics are software-demonstration values; "
            "the full 1,103-test-example frozen metrics remain the thesis evidence."
        )

        return (
            result[
                "answer"
            ],
            result[
                "latency_ms"
            ],
            result[
                "peak_vram_gb"
            ],
            result[
                "load_seconds"
            ],
            result[
                "prompt"
            ],
            status
        )

    except Exception as e:

        traceback.print_exc()

        return (
            "",
            0.0,
            0.0,
            0.0,
            "",
            f"Live inference error: {type(e).__name__}: {e}"
        )


print("✓ Dashboard helpers ready.")

✓ Dashboard helpers ready.


## Cell 21 — Build the full combined POC

In [ ]:
default_question, default_context, default_info = (
    load_question_details(
        QUESTION_LABELS[0]
    )
)

with gr.Blocks(
    title=
        "MR604 Full-Phase Corrected Context POC"
) as demo:

    gr.Markdown(
        "# MR604 Full-Phase Corrected Course-Logistics QA POC\n"
        "Six corrected PEFT conditions + two Qwen context references + "
        "deployment analysis + context-aware live saved-adapter QA."
    )

    gr.Markdown(
        "**Research evidence rule:** frozen metrics from the full 1,103-example "
        "corrected test evaluation are authoritative. "
        "The live QA panel is a software proof-of-concept."
    )


    # ------------------------------------------------------------
    # TAB 1 — Research evidence
    # ------------------------------------------------------------

    with gr.Tab(
        "Research Evidence"
    ):

        selected_condition = gr.Dropdown(
            choices=
                list(
                    LABEL_TO_KEY.keys()
                ),
            value=
                DEFAULT_LABEL,
            label=
                "Corrected PEFT condition"
        )

        selected_summary = gr.Markdown(
            selected_condition_summary(
                DEFAULT_LABEL
            )
        )

        gr.Markdown(
            "### Six corrected context-aware PEFT conditions"
        )

        gr.Dataframe(
            value=
                SLM_RESULTS.drop(
                    columns=[
                        "Key"
                    ]
                ).round(3),
            interactive=
                False
        )

        gr.Markdown(
            "### Corrected Qwen prompting references"
        )

        gr.Dataframe(
            value=
                QWEN_RESULTS.round(
                    3
                ),
            interactive=
                False
        )

        gr.Markdown(
            "### RQ2 paired LoRA vs QLoRA"
        )

        gr.Dataframe(
            value=
                RQ2_PAIRED.round(
                    3
                ),
            interactive=
                False
        )

        gr.Markdown(
            "### RQ2 seven native question types"
        )

        gr.Dataframe(
            value=
                RQ2_TYPES.round(
                    3
                ),
            interactive=
                False
        )


    # ------------------------------------------------------------
    # TAB 2 — Deployment
    # ------------------------------------------------------------

    with gr.Tab(
        "Deployment Analysis"
    ):

        priority = gr.Dropdown(
            choices=
                PRIORITIES,
            value=
                "Balanced deployment",
            label=
                "Institutional priority"
        )

        deployment_panel = gr.Markdown(
            deployment_summary(
                "Balanced deployment"
            )
        )

        apply_priority = gr.Button(
            "Apply criterion"
        )

        gr.Markdown(
            "### Pareto and 50/25/25 secondary score"
        )

        gr.Dataframe(
            value=
                DEPLOY[
                    [
                        "Rank",
                        "Condition",
                        "F1_percent",
                        "Avg_Latency_ms",
                        "Peak_Eval_VRAM_GB",
                        "Peak_Train_VRAM_GB",
                        "Pareto_Efficient",
                        "Deployment_Score"
                    ]
                ].round(
                    3
                ),
            interactive=
                False
        )


    # ------------------------------------------------------------
    # TAB 3 — Context-aware live QA
    # ------------------------------------------------------------

    with gr.Tab(
        "Live Context QA"
    ):

        gr.Markdown(
            f"**Runtime GPU:** {GPU_NAME}. "
            "The live question context is retrieved with the same BM25 design "
            "used by the corrected experiments."
        )

        live_model = gr.Dropdown(
            choices=
                list(
                    LABEL_TO_KEY.keys()
                ),
            value=
                DEFAULT_LABEL,
            label=
                "Corrected saved adapter"
        )

        question_selector = gr.Dropdown(
            choices=
                QUESTION_LABELS,
            value=
                QUESTION_LABELS[0],
            label=
                "Fixed SyllabusQA test example"
        )

        live_question = gr.Textbox(
            value=
                default_question,
            lines=
                3,
            label=
                "Question",
            interactive=
                False
        )

        example_info = gr.Textbox(
            value=
                default_info,
            lines=
                5,
            label=
                "Dataset / reference information",
            interactive=
                False
        )

        retrieved_context = gr.Textbox(
            value=
                default_context,
            lines=
                10,
            label=
                "BM25-retrieved syllabus context",
            interactive=
                False
        )

        run_button = gr.Button(
            "Run corrected saved adapter",
            variant=
                "primary"
        )

        live_answer = gr.Textbox(
            lines=
                8,
            label=
                "Generated answer",
            interactive=
                False
        )

        with gr.Row():

            live_latency = gr.Number(
                label=
                    "Generation latency (ms)",
                interactive=
                    False
            )

            live_vram = gr.Number(
                label=
                    "Peak live VRAM (GB)",
                interactive=
                    False
            )

            load_seconds = gr.Number(
                label=
                    "Model load time (s)",
                interactive=
                    False
            )

        exact_prompt = gr.Textbox(
            lines=
                10,
            label=
                "Exact corrected prompt",
            interactive=
                False
        )

        live_status = gr.Textbox(
            lines=
                3,
            label=
                "Status",
            interactive=
                False
        )


    selected_condition.change(
        fn=
            selected_condition_summary,
        inputs=
            selected_condition,
        outputs=
            selected_summary
    )

    apply_priority.click(
        fn=
            deployment_summary,
        inputs=
            priority,
        outputs=
            deployment_panel
    )

    question_selector.change(
        fn=
            load_question_details,
        inputs=
            question_selector,
        outputs=[
            live_question,
            retrieved_context,
            example_info
        ]
    )

    run_button.click(
        fn=
            run_live,
        inputs=[
            live_model,
            live_question,
            retrieved_context
        ],
        outputs=[
            live_answer,
            live_latency,
            live_vram,
            load_seconds,
            exact_prompt,
            live_status
        ]
    )


print("✓ Full updated combined POC built.")

✓ Full updated combined POC built.


## Cell 22 — Launch

In [ ]:
demo.launch(
    share=True,
    show_error=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://96cc51cc6c76f7709c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import files
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/MR603_Thesis")
RESULTS_DIR = BASE_DIR / "results"
POC_DIR = RESULTS_DIR / "full_phase_poc_context"

files_to_download = [
    POC_DIR / "RQ2_QuestionType_CONTEXT.csv",
    POC_DIR / "RQ3_Deployment_CONTEXT.csv",
    RESULTS_DIR / "REF_qwen14b_2shot_CONTEXT.json",
]

for f in files_to_download:
    if f.exists():
        print(f"Downloading: {f.name}")
        files.download(str(f))
    else:
        print(f"NOT FOUND: {f}")

Downloading: RQ2_QuestionType_CONTEXT.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: RQ3_Deployment_CONTEXT.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: REF_qwen14b_2shot_CONTEXT.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>